<a href="https://colab.research.google.com/github/faiz0407/Facial-Reconstruction-using-VAE/blob/main/VAEbasedImageReconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Conv2DTranspose, Reshape, Lambda, Activation, LeakyReLU
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
from glob import glob

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q kaggle

# Upload kaggle.json (you'll be prompted to upload)
from google.colab import files
files.upload()

# Move kaggle.json to the correct location
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download CelebA dataset
!kaggle datasets download -d jessicali9530/celeba-dataset

# Unzip the dataset
!unzip -q celeba-dataset.zip -d celeba


In [ ]:
WEIGHTS_FOLDER = '/content/drive/MyDrive/weights'
DATA_FOLDER = '/content/celeba/img_align_celeba/img_align_celeba'
Z_DIM = 200

if not os.path.exists(WEIGHTS_FOLDER):
  os.makedirs(os.path.join(WEIGHTS_FOLDER,"AE"))
  os.makedirs(os.path.join(WEIGHTS_FOLDER,"VAE"))

filenames = np.array(glob(os.path.join(DATA_FOLDER, '*.jpg')))
NUM_IMAGES = len(filenames)
print("Total number of images : " + str(NUM_IMAGES))

In [ ]:
def build_decoder(test=False, out_size=(128, 128)):
    def decoder(path):
        img = file_bytes = tf.io.read_file(path)
        img = tf.image.decode_jpeg(file_bytes, channels=3)
        img = tf.image.resize(img, (128, 128))
        img = tf.cast(img, tf.float32) / 255.0
        return img
    def decoder_train(path):
        return decoder(path), decoder(path)

    return decoder if test else decoder_train

def build_dataset(paths, test=False, shuffle=1, batch_size=1):
    AUTO = tf.data.experimental.AUTOTUNE
    decoder = build_decoder(test)

    dset = tf.data.Dataset.from_tensor_slices(paths)
    dset = dset.map(decoder, num_parallel_calls=AUTO)

    dset = dset.shuffle(shuffle)
    dset = dset.batch(batch_size)
    return dset

In [ ]:
train_paths, valid_paths, _, _ = train_test_split(filenames, filenames, test_size=0.2, shuffle=True)

train_dataset = build_dataset(train_paths, batch_size=128)
valid_dataset = build_dataset(valid_paths, batch_size=128)

In [ ]:
class ConvAutoencoder:
    def __init__(self):
        self.input_dim = (128,128,3)
        self.batch_size = 512
        self.latentDim = 200
        self.z_dim = 200 # Dimension of the latent vector (z)
        self.autoencoder_model = None
        self.encoder_model = None
        self.decoder_model = None


    def build(self):
        inputs = Input(shape = self.input_dim)
        x = inputs

        filters=(32, 64, 64, 64)

        for index, f in enumerate(filters):
            x = Conv2D(f, (3,3), strides=2, padding="same", name="conv2dtranspose_" + str(index))(x)
            x = LeakyReLU()(x)

        volumeSize = K.int_shape(x)
        x = Flatten()(x)
        latent = Dense(self.latentDim)(x)
        self.encoder_model = Model(inputs, latent, name = "encoder")

        latentInputs = Input(shape=(self.latentDim,))
        x = Dense(np.prod(volumeSize[1:]))(latentInputs)
        x = Reshape((volumeSize[1], volumeSize[2], volumeSize[3]))(x)

        for f in [64, 64, 32]:
            x = Conv2DTranspose(f, (3, 3), strides=2, padding="same")(x)
            x = LeakyReLU()(x)

        x = Conv2DTranspose(3, (3, 3), strides=2, padding="same")(x)
        outputs = Activation("sigmoid")(x)
        self.decoder_model = Model(latentInputs, outputs, name="decoder")
        self.autoencoder_model = Model(inputs, self.decoder_model(self.encoder_model(inputs)),name="autoencoder")

        return None

    def get_encoder(self):
        if self.encoder_model is not None:
            return self.encoder_model
        else:
            print("Encoder model has not been defined!")
            return None

    def get_decoder(self):
        if self.decoder_model is not None:
            return self.decoder_model
        else:
            print("Decoder model has not been defined!")
            return None

    def get_autoencoder(self):
        if self.autoencoder_model is not None:
            return self.autoencoder_model
        else:
            print("Autoencoder model has not been defined!")
            return None



In [ ]:
model = ConvAutoencoder()
model.build()
encoder = model.get_encoder()
decoder = model.get_decoder()
autoencoder = model.get_autoencoder()
encoder.summary()
decoder.summary()
autoencoder.summary()

In [ ]:
LEARNING_RATE = 0.0005
N_EPOCHS = 20

optimizer = Adam(learning_rate = LEARNING_RATE)

def r_loss(y_true, y_pred):
    return K.mean(K.square(y_true - y_pred), axis = [1,2,3])

autoencoder.compile(optimizer=optimizer, loss = r_loss)

checkpoint_ae_best = ModelCheckpoint(os.path.join(WEIGHTS_FOLDER, 'AE/autoencoder_best_weights.h5'),
                                     monitor='val_loss',
                                     mode='min',
                                     save_best_only=True,
                                     save_weights_only = False,
                                     verbose=1)

checkpoint_ae_last = ModelCheckpoint(os.path.join(WEIGHTS_FOLDER, 'AE/autoencoder_last_weights.h5'),
                                     monitor='val_loss',
                                     mode='min',
                                     save_best_only=True,
                                     save_weights_only = False,
                                     verbose=1)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint

# Sampling layer for reparameterization trick
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon
    def get_config(self):
        config = super(Sampling, self).get_config()
        return config

# Variational Autoencoder
class VAE(keras.Model):
    def __init__(self, input_dim, z_dim, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = self.build_encoder(input_dim, z_dim)
        self.decoder = self.build_decoder(input_dim, z_dim)
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    def build_encoder(self, input_dim, z_dim):
        encoder_inputs = keras.Input(shape=input_dim)
        x = layers.Conv2D(32, 3, strides=2, padding='same', activation='leaky_relu')(encoder_inputs)
        x = layers.Conv2D(64, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Conv2D(64, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Conv2D(64, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Flatten()(x)
        x = layers.Dense(16, activation='leaky_relu')(x)
        z_mean = layers.Dense(z_dim, name="z_mean")(x)
        z_log_var = layers.Dense(z_dim, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    def build_decoder(self, input_dim, z_dim):
        decoder_inputs = keras.Input(shape=(z_dim,))
        x = layers.Dense(16, activation='leaky_relu')(decoder_inputs)
        x = layers.Dense(np.prod((8, 8, 64)), activation='leaky_relu')(x)
        x = layers.Reshape((8, 8, 64))(x)
        x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='leaky_relu')(x)
        x = layers.Conv2DTranspose(3, 3, strides=2, padding='same', activation='sigmoid')(x)
        return keras.Model(decoder_inputs, x, name="decoder")

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        return self.decoder(z)

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]  # Only input images needed

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction loss (binary cross-entropy)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)
                )
            )

            # KL divergence
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

# Parameters
input_dim = (128, 128, 3)
z_dim = 200
learning_rate = 0.0005

# Instantiate and compile VAE
vae = VAE(input_dim, z_dim)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss=lambda y_true, y_pred: 0.0)  # Dummy loss

# Print summaries
vae.encoder.summary()
vae.decoder.summary()

In [ ]:
VAE_N_EPOCHS = 10
WEIGHTS_FOLDER = "/content/drive/MyDrive/practicum/weights"

# Callbacks
checkpoint_vae_best = ModelCheckpoint(
    WEIGHTS_FOLDER + '/VAE/vae_best_model.weights.h5',
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

checkpoint_vae_last = ModelCheckpoint(
    WEIGHTS_FOLDER + '/VAE/vae_last_model.weights.h5',
    monitor='val_loss',
    mode='min',
    save_best_only=False,
    save_weights_only=True,
    verbose=1
)

# Train the model
vae.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=VAE_N_EPOCHS,
    callbacks=[checkpoint_vae_best, checkpoint_vae_last]
)



In [ ]:
vae.save("vae.keras", save_format="keras")


In [ ]:
test_dataset = build_dataset(valid_paths, test=True)
vae.load_weights(WEIGHTS_FOLDER + '/VAE/vae_last_model.weights.h5')


In [ ]:
data = list(test_dataset.take(20))

fig = plt.figure(figsize=(30, 10))
for n in range(0, 20, 2):
    image = vae.predict(data[n])

    plt.subplot(2, 10, n + 1)
    plt.imshow(np.squeeze(data[n]))
    plt.title('original image')

    plt.subplot(2, 10, n + 2)
    plt.imshow(np.squeeze(image))
    plt.title('reconstruct')

plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import cv2

def process_image_through_vae(image, vae_model):
    image = cv2.resize(image, (128, 128))        # Resize here
    image = image.astype('float32') / 255.0       # Normalize
    image = np.expand_dims(image, axis=0)         # Add batch dimension
    output = vae_model(image)
    output = output.numpy()[0]                    # Remove batch dimension
    output = np.clip(output * 255, 0, 255).astype('uint8')  # Denormalize
    return output




In [ ]:
# Load an image manually (example)
img = cv2.imread('/content/12 Adorable Teen Singers, Then And Now.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Call the function
output_img = process_image_through_vae(img, vae_model=vae)

# Visualize
import matplotlib.pyplot as plt
plt.imshow(output_img)
plt.axis('off')
plt.show()
